# Task 5 ensemble sweep

Which of the trained sub-models should the Task 5 blend combine, and with which blender settings?

**Run after the training:** `submission/task5_ensemble.ipynb` stores every sub-model in `trained_models/task5/`. This notebook only
uses the stored validation predictions, so nothing is retrained. Sub-models that are not stored yet are left out and printed.
Run All runs every stage. Run time: every blend is a LightGBM pair blender, so expect seconds to minutes per blend.

**How a composition is scored:** the out-of-fold accuracy on the validation pairs (higher is better), the way the submitted
LightGBM pair blender is scored, averaged over 2 different splits of the validation pairs into folds.
Two compositions are compared pair by pair: change +- standard error. A change within about two standard errors is noise.

1. **Single models** and the blend of all
2. **Drop one:** the blend without each model -> how much it adds on top of all the others
3. **Backward elimination:** drop the least useful model again and again, as long as the blend stays within `tolerance` of the
   blend of all.
4. **Blender settings** on the selected models, compared with the current blender
5. **Recommendation** + report in `output/task5_ensemble/ensemble_<start time>.json`

**Apply the result:** copy the printed `TASK5_PAIR_BLEND_MODEL_FLAGS` into `lib/config.py` (and the printed blender setting changes,
if any, into `TASK5_PAIR_BLENDER`),
then rerun the blend cells of `task5_ensemble.ipynb`.

Every blend is a LightGBM classifier on both orders of every pair, so one blend takes a while (several fits per fold);
2 fold seeds keep the run time manageable. Dropping a model mainly saves its training time (EASE needs ~20 GB RAM).

In [ ]:
%load_ext autoreload
# --full: reload whole modules. The default only re-runs the changed lines of a module, so a setting computed from an
# edited one (e.g. TASK1_BLENDED_MODELS from TASK1_BLEND_MODEL_FLAGS in lib/config.py) would keep its old value
%autoreload 2 --full

# ensemble_search.py lives in this folder and imports assignment1.py / lib/ from ../submission
from dataclasses import replace

from ensemble_search import (PairBlendScorer, backward_elimination, blend_units, blender_comparison, drop_one_table,
                             load_stored_sub_models, print_recommendation, save_ensemble_report, single_model_table)

import assignment1
import lib.config as model_configs        # settings read when a cell runs -> edits of config.py apply without a restart

In [ ]:
# candidates: every sub-model with a flag in lib/config.py (True or False -> a left-out model can come back)
candidate_models = tuple(model_configs.TASK5_PAIR_BLEND_MODEL_FLAGS)
linked_models = []                              # no models that only go together
fold_seeds = (0, 1)
tolerance = 0.0005                              # accuracy the selected blend may lose against the blend of all

# stage 4: the first entry is the reference the others are compared with
blender_candidates = {
    "LightGBM (lib/config.py)": model_configs.TASK5_PAIR_BLENDER,
    "no time features": replace(model_configs.TASK5_PAIR_BLENDER, use_time_features=False),
    "15 leaves": replace(model_configs.TASK5_PAIR_BLENDER, lightgbm_num_leaves=15),
    "63 leaves": replace(model_configs.TASK5_PAIR_BLENDER, lightgbm_num_leaves=63),
    "min. 50 pairs per leaf": replace(model_configs.TASK5_PAIR_BLENDER, lightgbm_min_child_samples=50),
    "min. 1000 pairs per leaf": replace(model_configs.TASK5_PAIR_BLENDER, lightgbm_min_child_samples=1000),
    "800 trees, learning rate 0.025": replace(model_configs.TASK5_PAIR_BLENDER, lightgbm_n_estimators=800, lightgbm_learning_rate=0.025),
}

In [ ]:
task5 = assignment1.load_task5()
sub_model_results = load_stored_sub_models("task5", candidate_models)
stored_models = tuple(sub_model_results)
units = blend_units(stored_models, linked_models)
scorer = PairBlendScorer(task5, sub_model_results, fold_seeds)
report = {"task": "task5", "stored_models": stored_models, "missing_models": [m for m in candidate_models if m not in stored_models],
          "fold_seeds": fold_seeds, "tolerance": tolerance, "base_blender": model_configs.TASK5_PAIR_BLENDER}
print(f"{len(stored_models)} stored sub-models, {len(units)} units: {units}")

## 1. Single models and the blend of all

In [ ]:
report["single"] = single_model_table(scorer, model_configs.TASK5_PAIR_BLENDER, stored_models)

## 2. Drop one
Change of the blend without each model (negative accuracy change = the blend gets worse without it = the model helps). Most useful first.

In [ ]:
report["drop_one"] = drop_one_table(scorer, model_configs.TASK5_PAIR_BLENDER, units)

## 3. Backward elimination
Every step drops the model whose removal hurts least; it stops before the blend would lose more than `tolerance`
against the blend of all models.

In [ ]:
elimination = backward_elimination(scorer, model_configs.TASK5_PAIR_BLENDER, units, tolerance)
report["elimination"] = elimination

## 4. Blender settings on the selected models
Each blender compared with the first entry (the current one). Another blender is only recommended if it beats it by more than
two standard errors.

In [ ]:
blenders = blender_comparison(scorer, blender_candidates, elimination["selected_models"])
report["blenders"] = blenders

## 5. Recommendation
Copy into `lib/config.py` (Task 5 section), then rerun the blend cells of `task5_ensemble.ipynb`.

In [ ]:
report["recommendation"] = print_recommendation(scorer, blender_candidates, elimination, blenders, "TASK5_PAIR_BLEND_MODEL_FLAGS",
                                                model_configs.TASK5_PAIR_BLEND_MODEL_FLAGS, stored_models,
                                                {"TASK5_PAIR_BLENDER": model_configs.TASK5_PAIR_BLENDER})
if report["missing_models"]:
    print(f"(not part of this search, not stored yet: {', '.join(report['missing_models'])})")
save_ensemble_report("task5", report)